### A notebook to centralize the data collection mechanisms

### Overall Purpose of the notebook
# Build curated drug-drug interaction (DDI) datasets from raw negative and adverse interaction files.
# Enrich each drug with features such as SMILES, ATC codes, target identifiers, FASTA sequences, GO terms, Pfam domains, and pathway-based neighbor proteins.
# Filter the datasets down to only drug pairs where both drugs have a complete feature set.
# Save the final curated datasets as Parquet files.

In [ ]:
# A. The Monotonic Ladder (Section 1.2)The output columns currently say target_uniprot_ids. 
# This implies you are currently only running Level 0 ($L_0$) (Pure Pharmacodynamics).To execute $L_1$ and $L_2$, you must create new columns in your DataFrame before feeding it to the script. 
# For example:L1_uniprot_ids = target_uniprot $\cup$ enzyme_uniprot $\cup$ transporter_uniprotL2_uniprot_ids = L1_uniprot_ids $\cup$ carrier_uniprotYou would then pass these new $L_1$ and $L_2$ columns into the feature_mappings argument of the script to measure how similarity changes as the biological definition widens.B. 
# Splitting the Gene Ontology (Section 3.2)Your framework specifies that GO terms should be kept separable by ontology to capture distinct similarity flavors:$GO_{MF}$ (Molecular Function)$GO_{BP}$ (Biological Process)$GO_{CC}$ (Cellular Component)

In [1]:
# Imports

import pandas as pd
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
original_negative_dataset = pd.read_csv("C:\\Users\\ashto\\OneDrive - Eastern Connecticut State University\\Project 5. Data\\raw\\negative_ddi_samples.csv")
original_adverse_dataset = pd.read_csv("C:\\Users\\ashto\\OneDrive - Eastern Connecticut State University\\Project 5. Data\\raw\\removal_of_positive_ddis\\drugbank_approved_small_1113772_ddi_pairs_positive_removed.csv")

In [3]:
# ...existing code...
# Cell 1: Imports & load raw pair datasets

import pandas as pd
import numpy as np
import os
import zipfile
import time
import requests
from pathlib import Path
import xml.etree.ElementTree as ET

original_negative_dataset = pd.read_csv(
    r"C:\Users\ashto\OneDrive - Eastern Connecticut State University\Project 5. Data\raw\negative_ddi_samples.csv"
)
original_adverse_dataset = pd.read_csv(
    r"C:\Users\ashto\OneDrive - Eastern Connecticut State University\Project 5. Data\raw\drugbank_approved_small_2369_1129743_ddi_pairs.csv"
)

def norm_id(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().upper()

def infer_pair_cols(df):
    candidates = [
        ("drug1", "drug2"),
        ("drug_1", "drug_2"),
        ("drugbank_id_1", "drugbank_id_2"),
        ("drug1_id", "drug2_id"),
        ("left_drugbank_id", "right_drugbank_id"),
        ("Drug1_ID", "Drug2_ID"),
    ]
    cols = {c.lower(): c for c in df.columns}
    for a, b in candidates:
        if a.lower() in cols and b.lower() in cols:
            return cols[a.lower()], cols[b.lower()]
    raise ValueError(f"Could not infer pair columns. Available: {list(df.columns)}")

neg_d1, neg_d2 = infer_pair_cols(original_negative_dataset)
adv_d1, adv_d2 = infer_pair_cols(original_adverse_dataset)

needed_ids = set(
    pd.concat([
        original_negative_dataset[neg_d1], original_negative_dataset[neg_d2],
        original_adverse_dataset[adv_d1], original_adverse_dataset[adv_d2]
    ], axis=0).dropna().map(norm_id).unique()
)

print(f"Negative pair cols: {neg_d1}, {neg_d2} | rows: {len(original_negative_dataset)}")
print(f"Adverse pair cols : {adv_d1}, {adv_d2} | rows: {len(original_adverse_dataset)}")
print(f"Unique DrugBank IDs needed: {len(needed_ids)}")

Negative pair cols: drug1_id, drug2_id | rows: 1113772
Adverse pair cols : drug1_id, drug2_id | rows: 1129743
Unique DrugBank IDs needed: 4563


In [4]:
# ...existing code...
# Cell 2: DrugBank XML parsing helpers (correct XSD structure)

def _extract_smiles(elem, ns):
    calc_props = elem.find(f"{ns}calculated-properties")
    if calc_props is not None:
        for prop in calc_props.findall(f"{ns}property"):
            kind_el = prop.find(f"{ns}kind")
            if kind_el is not None and (kind_el.text or "").strip().upper() == "SMILES":
                val_el = prop.find(f"{ns}value")
                if val_el is not None and val_el.text:
                    return val_el.text.strip()
    exp_props = elem.find(f"{ns}experimental-properties")
    if exp_props is not None:
        for prop in exp_props.findall(f"{ns}property"):
            kind_el = prop.find(f"{ns}kind")
            if kind_el is not None and (kind_el.text or "").strip().upper() == "SMILES":
                val_el = prop.find(f"{ns}value")
                if val_el is not None and val_el.text:
                    return val_el.text.strip()
    return np.nan


def _extract_atc_list(elem, ns):
    codes = set()
    atc_block = elem.find(f"{ns}atc-codes")
    if atc_block is not None:
        for a in atc_block.findall(f"{ns}atc-code"):
            code = a.get("code")
            if code:
                codes.add(code)
    return sorted(codes)


def _extract_targets(elem, ns):
    """
    Returns dict:
      - uniprot_ids
      - fasta_sequences
      - other_ids (gene names, polypeptide names, target names, non-uniprot external ids)
      - actions
    """
    uniprot_ids = set()
    fasta_sequences = set()
    other_ids = set()
    actions = set()

    targets_block = elem.find(f"{ns}targets")
    if targets_block is None:
        return {"uniprot_ids": [], "fasta_sequences": [], "other_ids": [], "actions": []}

    for t in targets_block.findall(f"{ns}target"):
        t_name_el = t.find(f"{ns}name")
        if t_name_el is not None and t_name_el.text:
            other_ids.add(t_name_el.text.strip())

        actions_block = t.find(f"{ns}actions")
        if actions_block is not None:
            for act in actions_block.findall(f"{ns}action"):
                if act.text:
                    actions.add(act.text.strip())

        polypeptide_block = t.find(f"{ns}polypeptide")
        if polypeptide_block is not None:
            pid = (polypeptide_block.get("id") or "").strip()
            source = (polypeptide_block.get("source") or "").strip().lower()

            if pid:
                if source in ("swiss-prot", "uniprotkb", ""):
                    uniprot_ids.add(pid)
                else:
                    other_ids.add(pid)

            gene_el = polypeptide_block.find(f"{ns}gene-name")
            if gene_el is not None and gene_el.text:
                other_ids.add(gene_el.text.strip())

            pname_el = polypeptide_block.find(f"{ns}name")
            if pname_el is not None and pname_el.text:
                other_ids.add(pname_el.text.strip())

            seq_el = polypeptide_block.find(f"{ns}amino-acid-sequence")
            if seq_el is not None and seq_el.text:
                lines = seq_el.text.strip().splitlines()
                seq_body = "".join(lines[1:]) if lines and lines[0].startswith(">") else "".join(lines)
                if seq_body:
                    fasta_sequences.add(seq_body.strip())

            ext_block = polypeptide_block.find(f"{ns}external-identifiers")
            if ext_block is not None:
                for ext in ext_block.findall(f"{ns}external-identifier"):
                    res_el = ext.find(f"{ns}resource")
                    id_el = ext.find(f"{ns}identifier")
                    res = (res_el.text or "").strip().lower() if res_el is not None else ""
                    ident = (id_el.text or "").strip() if id_el is not None else ""
                    if not ident:
                        continue
                    if res == "uniprotkb":
                        uniprot_ids.add(ident)
                    else:
                        other_ids.add(ident)

    return {
        "uniprot_ids": sorted(uniprot_ids),
        "fasta_sequences": sorted(fasta_sequences),
        "other_ids": sorted(other_ids),
        "actions": sorted(actions),
    }


def _extract_groups(elem, ns):
    groups = []
    g_block = elem.find(f"{ns}groups")
    if g_block is not None:
        for g in g_block.findall(f"{ns}group"):
            if g.text:
                groups.append(g.text.strip().lower())
    return sorted(set(groups))

In [5]:
# ...existing code...
# Cell 3: Full DrugBank parse — include APPROVED and UNAPPROVED small molecules (not withdrawn)

DRUGBANK_XML_PATH = r"C:\Users\ashto\ddi-prediction\data\raw\drugbank_full_database_V5.1.14.zip"

def _parse_drugbank_stream(stream, wanted_ids):
    records = []
    total_drugs = 0
    matched = 0

    for _, elem in ET.iterparse(stream, events=("end",)):
        if not elem.tag.endswith("drug"):
            continue

        ns = elem.tag.split("}")[0] + "}" if "}" in elem.tag else ""
        total_drugs += 1

        dbid = None
        for x in elem.findall(f"{ns}drugbank-id"):
            if x.get("primary") == "true":
                dbid = (x.text or "").strip().upper()
                break
        if dbid is None:
            first = elem.find(f"{ns}drugbank-id")
            if first is not None and first.text:
                dbid = first.text.strip().upper()

        if (dbid is None) or (dbid not in wanted_ids):
            elem.clear()
            continue

        name_el = elem.find(f"{ns}name")
        drug_name = name_el.text.strip() if name_el is not None and name_el.text else None

        # -----------------------------
        # FILTER: small molecule + not withdrawn (approval NOT required here)
        # -----------------------------
        groups = [
            g.text for g in elem.findall(f"{ns}groups/{ns}group")
            if g.text
        ]

        if not (
            elem.get("type") == "small molecule"
            and "withdrawn" not in groups
        ):
            elem.clear()
            continue

        drug_type = (elem.get("type") or "").strip().lower()
        is_small_molecule = drug_type == "small molecule"
        is_approved = "approved" in groups
        is_withdrawn = "withdrawn" in groups

        atc_list = _extract_atc_list(elem, ns)
        targets = _extract_targets(elem, ns)

        records.append({
            "drugbank_id": dbid,
            "drug_name": drug_name,
            "drug_type": drug_type,
            "is_small_molecule": is_small_molecule,
            "smiles": _extract_smiles(elem, ns),
            "atc_codes_list": atc_list,
            "target_uniprot_ids": targets["uniprot_ids"],
            "target_fasta_sequences": targets["fasta_sequences"],
            "target_other_ids": targets["other_ids"],
            "target_actions": targets["actions"],
            "drug_groups": groups,
            "is_approved": bool(is_approved),
            "is_withdrawn": bool(is_withdrawn),
        })

        matched += 1
        if matched % 200 == 0:
            print(f"Inspected {total_drugs} | Matched {matched}")

        elem.clear()

    print(f"\nFinished. Total inspected: {total_drugs} | Matched: {matched}")
    return pd.DataFrame(records).drop_duplicates(subset=["drugbank_id"])


def parse_drugbank_full(xml_or_zip_path, wanted_ids):
    p = Path(xml_or_zip_path)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p, "r") as zf:
            xml_members = [n for n in zf.namelist() if n.lower().endswith(".xml")]
            if not xml_members:
                raise FileNotFoundError("No XML found inside zip.")
            print("Streaming:", xml_members[0])
            with zf.open(xml_members[0], "r") as f:
                return _parse_drugbank_stream(f, wanted_ids)
    else:
        with open(p, "rb") as f:
            return _parse_drugbank_stream(f, wanted_ids)


drugbank_df = parse_drugbank_full(DRUGBANK_XML_PATH, needed_ids)
print("\ndrugbank_df shape:", drugbank_df.shape)
print("is_small_molecule counts:\n", drugbank_df["is_small_molecule"].value_counts())
print("is_approved counts:\n", drugbank_df["is_approved"].value_counts())
drugbank_df.head()

Streaming: drugbank_full_database_V5.1.14.xml
Inspected 281054 | Matched 200
Inspected 282376 | Matched 400
Inspected 284766 | Matched 600
Inspected 286790 | Matched 800
Inspected 296186 | Matched 1000
Inspected 640939 | Matched 1200
Inspected 995535 | Matched 1400
Inspected 996426 | Matched 1600
Inspected 1002503 | Matched 1800
Inspected 1002950 | Matched 2000
Inspected 1004244 | Matched 2200
Inspected 1004902 | Matched 2400
Inspected 1005737 | Matched 2600
Inspected 1006112 | Matched 2800
Inspected 1006496 | Matched 3000
Inspected 1007586 | Matched 3200

Finished. Total inspected: 1014328 | Matched: 3345

drugbank_df shape: (3345, 13)
is_small_molecule counts:
 is_small_molecule
True    3345
Name: count, dtype: int64
is_approved counts:
 is_approved
True     1902
False    1443
Name: count, dtype: int64


,drugbank_id,drug_name,drug_type,is_small_molecule,smiles,atc_codes_list,target_uniprot_ids,target_fasta_sequences,target_other_ids,target_actions,drug_groups,is_approved,is_withdrawn
0,DB00006,Bivalirudin,small molecule,True,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,[B01AE06],[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...",[inhibitor],"[approved, investigational]",True,False
1,DB00014,Goserelin,small molecule,True,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,[L02AE03],"[P01148, P22888, P30968]",[MANSASPEQNQNHCSAINNSIPLMQGNLPTLTLSGKIRVTVTFFL...,"[183422, 254, 256, 903746, GNRH1, GNRHR, GNRHR...","[activator, agonist]","[approved, investigational]",True,False
2,DB00027,Gramicidin D,small molecule,True,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,[R02AB30],[P0AC13],[MKLFAQGTSLDLSHPHVMGILNVTPDSFSDGGTHNSLIDAVKHAN...,"[41273, DHPS_ECOLI, Dihydropteroate synthase, ...",[binder],"[approved, investigational]",True,False
3,DB00035,Desmopressin,small molecule,True,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,[H01BA02],"[P30518, P30559, P37288, P47901]",[MDSGPLWDANPTPRGTLSAPNATTPWLGRDEELAKVEIGVLATVL...,"[28418, 34765, 366, 367, 368, 369, 563982, 667...",[agonist],"[approved, investigational]",True,False
4,DB00080,Daptomycin,small molecule,True,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,[J01XX09],[P0AC13],[MKLFAQGTSLDLSHPHVMGILNVTPDSFSDGGTHNSLIDAVKHAN...,"[41273, Cytoplasmic membrane, DHPS_ECOLI, Dihy...","[binder, incorporation into and destabilization]","[approved, investigational]",True,False


In [6]:
# ...existing code...
# Cell 4: Restrict to small-molecule drugs (approved OR unapproved, just not withdrawn), then filter both pair datasets

sm_ids = set(
    drugbank_df.loc[
        (drugbank_df["is_small_molecule"] == True) & (drugbank_df["is_withdrawn"] == False),
        "drugbank_id"
    ]
)

print(f"Small-molecule (approved + unapproved, non-withdrawn) drugs available: {len(sm_ids)}")

def filter_pairs_to_ids(df, d1_col, d2_col, allowed_ids):
    out = df.copy()
    d1 = out[d1_col].astype(str).str.strip().str.upper()
    d2 = out[d2_col].astype(str).str.strip().str.upper()
    mask = d1.isin(allowed_ids) & d2.isin(allowed_ids)
    return out.loc[mask].copy()

negative_df = filter_pairs_to_ids(original_negative_dataset, neg_d1, neg_d2, sm_ids)
adverse_df  = filter_pairs_to_ids(original_adverse_dataset,  adv_d1, adv_d2, sm_ids)

print(f"Negative: {len(original_negative_dataset)} -> {len(negative_df)}")
print(f"Adverse : {len(original_adverse_dataset)} -> {len(adverse_df)}")

Small-molecule (approved + unapproved, non-withdrawn) drugs available: 3345
Negative: 1113772 -> 569934
Adverse : 1129743 -> 830623


In [7]:
# ...existing code...
# Cell 5: Attach per-drug features (smiles, atc list, target lists) to both pair datasets

feat_lookup = drugbank_df.set_index("drugbank_id")
feature_cols = ["smiles", "atc_codes_list", "target_uniprot_ids", "target_fasta_sequences", "target_other_ids"]

def attach_features(df, d1_col, d2_col, lookup_df, cols):
    out = df.copy()
    out["_d1"] = out[d1_col].astype(str).str.strip().str.upper()
    out["_d2"] = out[d2_col].astype(str).str.strip().str.upper()

    for col in cols:
        col_map = lookup_df[col].to_dict()
        out[f"{col}_1"] = out["_d1"].map(col_map)
        out[f"{col}_2"] = out["_d2"].map(col_map)

    return out.drop(columns=["_d1", "_d2"])

negative_feat_df = attach_features(negative_df, neg_d1, neg_d2, feat_lookup, feature_cols)
adverse_feat_df  = attach_features(adverse_df,  adv_d1, adv_d2, feat_lookup, feature_cols)

print("Negative feat shape:", negative_feat_df.shape)
print("Adverse  feat shape:", adverse_feat_df.shape)
negative_feat_df.head()

Negative feat shape: (569934, 15)
Adverse  feat shape: (830623, 16)


,drug1_id,drug1_name,drug2_id,drug2_name,description,smiles_1,smiles_2,atc_codes_list_1,atc_codes_list_2,target_uniprot_ids_1,target_uniprot_ids_2,target_fasta_sequences_1,target_fasta_sequences_2,target_other_ids_1,target_other_ids_2
0,DB06608,Tafenoquine,DB08899,Enzalutamide,No documented interaction,COC1=CC(C)=C2C(OC3=CC=CC(=C3)C(F)(F)F)=C(OC)C=...,CNC(=O)C1=C(F)C=C(C=C1)N1C(=S)N(C(=O)C1(C)C)C1...,[P01BA07],[L02BB04],[],[P10275],[],[MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAA...,[],"[178628, 628, ANDR_HUMAN, AR, Androgen recepto..."
4,DB06264,Tolperisone,DB14715,Cinazepam,No documented interaction,CC(CN1CCCCC1)C(=O)C1=CC=C(C)C=C1,OC(=O)CCC(=O)OC1N=C(C2=CC=CC=C2Cl)C2=CC(Br)=CC...,"[M02AX06, M03BX04]",[],[Q9UI33],[P14867],[MDDRCYPVIFPDERNFRPFTSDSLAAIEKRIAIQKEKKKSKDQTG...,[MRKSPGLSDCLWAWILLLSTLTGRSYGQPSLQDELKDNTTVFTRI...,"[586, 6572950, AF188679, HGNC:10583, SCN11A, S...","[31631, 404, GABA(A) Receptor, GABA(A) Recepto..."
7,DB05814,GPI-1485,DB13508,Cloranolol,No documented interaction,CCC(C)(C)C(=O)C(=O)N1CCC[C@H]1C(O)=O,CC(C)(C)NCC(O)COC1=CC(Cl)=CC=C1Cl,[],[C07AA27],[P62942],[],[MGVQVETISPGDGRTFPKRGQTCVVHYTGMLEDGKKFDSSRDRNK...,[],"[182628, 2609, FKB1A_HUMAN, FKBP1A, HGNC:3711,...",[]
9,DB01537,"4-Bromo-2,5-dimethoxyphenethylamine",DB13132,Artemisinin,No documented interaction,COC1=CC(Br)=C(OC)C=C1CCN,[H][C@@]1(C)CC[C@@]2([H])[C@@]([H])(C)C(=O)O[C...,[],"[P01BE01, P01BF07, P01BF08]","[P28223, P28335]",[Q08210],[MDILCEENTSLSSTTNSLMQLNDDTRLYSNDFNSGEANTSDAFNW...,[MISKLKPQFMFLPKKHILSYCRKDVLNLFEQKFYYTSKRKESNNM...,"[338028, 36431, 5-hydroxytryptamine receptor 2...","[46362265, CR382398, Dihydroorotate dehydrogen..."
10,DB01445,Bufotenine,DB05440,SRP 299,No documented interaction,CN(C)CCC1=CNC2=C1C=C(O)C=C2,NaN,[],[],"[P28221, P28223, P34969, P47898, P50406]",[],[MDILCEENTSLSSTTNSLMQLNDDTRLYSNDFNSGEANTSDAFNW...,[],"[10, 11, 1162924, 12, 177772, 1857143, 3, 3643...",[]


In [8]:
# ...existing code...
# Cell 6: Build a per-drug feature completeness table (one row per unique drug, not per pair)

def list_is_empty(x):
    if isinstance(x, list):
        return len(x) == 0
    return pd.isna(x)

drugbank_df["has_smiles"] = drugbank_df["smiles"].notna()
drugbank_df["has_atc"] = drugbank_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
drugbank_df["has_targets"] = (
    drugbank_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x)) |
    drugbank_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
drugbank_df["has_all_features"] = (
    drugbank_df["has_smiles"] & drugbank_df["has_atc"] & drugbank_df["has_targets"]
)

# Only look at drugs relevant to our filtered pair datasets
relevant_ids = set(
    pd.concat([
        negative_df[neg_d1], negative_df[neg_d2],
        adverse_df[adv_d1], adverse_df[adv_d2]
    ]).astype(str).str.strip().str.upper().unique()
)

per_drug_df = drugbank_df.loc[drugbank_df["drugbank_id"].isin(relevant_ids)].copy()

print(f"Relevant unique drugs: {len(relevant_ids)}")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

missing_smiles_ids = per_drug_df.loc[~per_drug_df["has_smiles"], "drugbank_id"].tolist()
missing_atc_ids = per_drug_df.loc[~per_drug_df["has_atc"], "drugbank_id"].tolist()
missing_targets_ids = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()

print(f"\nMissing SMILES : {len(missing_smiles_ids)}")
print(f"Missing ATC    : {len(missing_atc_ids)}")
print(f"Missing targets: {len(missing_targets_ids)}")

Relevant unique drugs: 3345
has_smiles          0.972795
has_atc             0.660688
has_targets         0.746487
has_all_features    0.505830
dtype: float64

Missing SMILES : 91
Missing ATC    : 1135
Missing targets: 848


In [ ]:
# ...existing code...
# Cell 7: Backfill missing SMILES from PubChem (by DrugBank ID via PubChem's xref, or by name)

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def pubchem_smiles_by_drugbank_xref(drugbank_id, timeout=10):
    """Try PubChem's xref lookup using DrugBank ID as the external identifier."""
    url = f"{PUBCHEM_BASE}/compound/xref/RegistryID/{drugbank_id}/property/CanonicalSMILES/JSON"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            props = data.get("PropertyTable", {}).get("Properties", [])
            if props:
                return props[0].get("CanonicalSMILES")
    except Exception:
        pass
    return None

def pubchem_smiles_by_name(drug_name, timeout=10):
    if not drug_name:
        return None
    url = f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(drug_name)}/property/CanonicalSMILES/JSON"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            props = data.get("PropertyTable", {}).get("Properties", [])
            if props:
                return props[0].get("CanonicalSMILES")
    except Exception:
        pass
    return None

pubchem_smiles_results = {}

for i, row in per_drug_df.loc[~per_drug_df["has_smiles"]].iterrows():
    dbid = row["drugbank_id"]
    name = row["drug_name"]

    smi = pubchem_smiles_by_drugbank_xref(dbid)
    if not smi:
        smi = pubchem_smiles_by_name(name)

    pubchem_smiles_results[dbid] = smi
    time.sleep(0.2)  # be polite to PubChem's rate limits

    if smi:
        print(f"[PubChem] {dbid} ({name}) -> found SMILES")

resolved_smiles_count = sum(1 for v in pubchem_smiles_results.values() if v)
print(f"\nPubChem resolved {resolved_smiles_count}/{len(pubchem_smiles_results)} missing SMILES")

In [ ]:
# ...existing code...
# Cell 8: Backfill missing ATC codes and target info from ChEMBL

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_molecule_id_by_name(drug_name, timeout=10):
    if not drug_name:
        return None
    url = f"{CHEMBL_BASE}/molecule/search.json"
    try:
        r = requests.get(url, params={"q": drug_name}, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            mols = data.get("molecules", [])
            if mols:
                return mols[0].get("molecule_chembl_id")
    except Exception:
        pass
    return None

def chembl_atc_codes(chembl_id, timeout=10):
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/molecule/{chembl_id}.json"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            atc_classes = data.get("atc_classifications", []) or []
            return sorted(set(atc_classes))
    except Exception:
        pass
    return []

def chembl_targets(chembl_id, timeout=10):
    """Return list of target ChEMBL IDs / UniProt accessions via mechanism endpoint."""
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/mechanism.json"
    try:
        r = requests.get(url, params={"molecule_chembl_id": chembl_id}, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            mechs = data.get("mechanisms", [])
            targets = set()
            for m in mechs:
                tgt = m.get("target_chembl_id")
                if tgt:
                    targets.add(tgt)
            return sorted(targets)
    except Exception:
        pass
    return []

chembl_atc_results = {}
chembl_target_results = {}

need_atc_or_targets = per_drug_df.loc[
    (~per_drug_df["has_atc"]) | (~per_drug_df["has_targets"])
]

for i, row in need_atc_or_targets.iterrows():
    dbid = row["drugbank_id"]
    name = row["drug_name"]

    chembl_id = chembl_molecule_id_by_name(name)

    if not row["has_atc"]:
        atc = chembl_atc_codes(chembl_id)
        chembl_atc_results[dbid] = atc
        if atc:
            print(f"[ChEMBL] {dbid} ({name}) -> ATC found: {atc}")

    if not row["has_targets"]:
        tgts = chembl_targets(chembl_id)
        chembl_target_results[dbid] = tgts
        if tgts:
            print(f"[ChEMBL] {dbid} ({name}) -> targets found: {tgts}")

    time.sleep(0.2)

print(f"\nChEMBL resolved ATC for {sum(1 for v in chembl_atc_results.values() if v)}/{len(chembl_atc_results)}")
print(f"ChEMBL resolved targets for {sum(1 for v in chembl_target_results.values() if v)}/{len(chembl_target_results)}")

[ChEMBL] DB00351 (Megestrol acetate) -> ATC found: ['G03AC05', 'G03DB02', 'L02AB01']
[ChEMBL] DB00460 (Verteporfin) -> targets found: ['CHEMBL2311221']
[ChEMBL] DB00823 (Ethynodiol diacetate) -> ATC found: ['G03DC06']
[ChEMBL] DB00832 (Phensuximide) -> targets found: ['CHEMBL2362995']
[ChEMBL] DB01013 (Clobetasol propionate) -> ATC found: ['D07AD01']
[ChEMBL] DB01049 (Ergoloid mesylate) -> ATC found: ['C04AE01', 'C04AE51']
[ChEMBL] DB01336 (Metocurine) -> ATC found: ['M03AA04']
[ChEMBL] DB01380 (Cortisone acetate) -> ATC found: ['H02AB10', 'S01BA03']
[ChEMBL] DB01481 (1-Testosterone) -> ATC found: ['G03BA03']
[ChEMBL] DB01529 (Dextromoramide) -> targets found: ['CHEMBL2095181']
[ChEMBL] DB01572 (Methyl-1-testosterone) -> ATC found: ['G03BA03']
[ChEMBL] DB01572 (Methyl-1-testosterone) -> targets found: ['CHEMBL1871']
[ChEMBL] DB01583 (Liotrix) -> ATC found: ['H03AA02']
[ChEMBL] DB02247 (Hydrolyzed Cephalothin) -> ATC found: ['J01DB03']
[ChEMBL] DB02699 (4-Oxoretinol) -> ATC found: ['A16

KeyboardInterrupt: 

In [ ]:
# ...existing code...
# Cell 8: Backfill missing ATC codes and target info from ChEMBL — PARALLELIZED

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
CHEMBL_MAX_WORKERS = 8

def chembl_molecule_id_by_name(drug_name, timeout=10):
    if not drug_name:
        return None
    url = f"{CHEMBL_BASE}/molecule/search.json"
    try:
        r = requests.get(url, params={"q": drug_name}, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            mols = data.get("molecules", [])
            if mols:
                return mols[0].get("molecule_chembl_id")
    except Exception:
        pass
    return None

def chembl_atc_codes(chembl_id, timeout=10):
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/molecule/{chembl_id}.json"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            atc_classes = data.get("atc_classifications", []) or []
            return sorted(set(atc_classes))
    except Exception:
        pass
    return []

def chembl_targets(chembl_id, timeout=10):
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/mechanism.json"
    try:
        r = requests.get(url, params={"molecule_chembl_id": chembl_id}, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            mechs = data.get("mechanisms", [])
            targets = set()
            for m in mechs:
                tgt = m.get("target_chembl_id")
                if tgt:
                    targets.add(tgt)
            return sorted(targets)
    except Exception:
        pass
    return []

def _resolve_chembl(dbid, name, need_atc, need_targets):
    chembl_id = chembl_molecule_id_by_name(name)
    atc = chembl_atc_codes(chembl_id) if need_atc else None
    tgts = chembl_targets(chembl_id) if need_targets else None
    return dbid, atc, tgts

need_atc_or_targets = per_drug_df.loc[
    (~per_drug_df["has_atc"]) | (~per_drug_df["has_targets"]),
    ["drugbank_id", "drug_name", "has_atc", "has_targets"]
]

chembl_atc_results = {}
chembl_target_results = {}

print(f"Resolving ATC/targets for {len(need_atc_or_targets)} drugs using {CHEMBL_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=CHEMBL_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _resolve_chembl,
            row.drugbank_id,
            row.drug_name,
            not row.has_atc,
            not row.has_targets
        ): row.drugbank_id
        for row in need_atc_or_targets.itertuples()
    }

    completed = 0
    for future in as_completed(futures):
        dbid, atc, tgts = future.result()

        if atc is not None:
            chembl_atc_results[dbid] = atc
            if atc:
                print(f"[ChEMBL] {dbid} -> ATC found: {atc}")

        if tgts is not None:
            chembl_target_results[dbid] = tgts
            if tgts:
                print(f"[ChEMBL] {dbid} -> targets found: {tgts}")

        completed += 1
        if completed % 100 == 0:
            print(f"  ... {completed}/{len(need_atc_or_targets)} processed")

print(f"\nChEMBL resolved ATC for {sum(1 for v in chembl_atc_results.values() if v)}/{len(chembl_atc_results)}")
print(f"ChEMBL resolved targets for {sum(1 for v in chembl_target_results.values() if v)}/{len(chembl_target_results)}")

Resolving ATC/targets for 1626 drugs using 8 parallel workers...
[ChEMBL] DB00460 -> targets found: ['CHEMBL2311221']
[ChEMBL] DB00351 -> ATC found: ['G03AC05', 'G03DB02', 'L02AB01']
[ChEMBL] DB00832 -> targets found: ['CHEMBL2362995']
[ChEMBL] DB00823 -> ATC found: ['G03DC06']
[ChEMBL] DB01049 -> ATC found: ['C04AE01', 'C04AE51']
[ChEMBL] DB01013 -> ATC found: ['D07AD01']
[ChEMBL] DB01336 -> ATC found: ['M03AA04']
[ChEMBL] DB01380 -> ATC found: ['H02AB10', 'S01BA03']
[ChEMBL] DB01481 -> ATC found: ['G03BA03']
[ChEMBL] DB01529 -> targets found: ['CHEMBL2095181']
  ... 100/1626 processed
[ChEMBL] DB01583 -> ATC found: ['H03AA02']
[ChEMBL] DB01572 -> ATC found: ['G03BA03']
[ChEMBL] DB01572 -> targets found: ['CHEMBL1871']
[ChEMBL] DB02247 -> ATC found: ['J01DB03']
[ChEMBL] DB02699 -> ATC found: ['A16AB12']
[ChEMBL] DB02699 -> targets found: ['CHEMBL3140341', 'CHEMBL3140342']
  ... 200/1626 processed
[ChEMBL] DB02968 -> ATC found: ['J01CE08']
[ChEMBL] DB03394 -> ATC found: ['A06AD15', 'A0

In [ ]:
# ...existing code...
# Cell 9: Merge external backfill results into per_drug_df

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

# Backfill SMILES
for dbid, smi in pubchem_smiles_results.items():
    if smi:
        per_drug_df.loc[dbid, "smiles"] = smi

# Backfill ATC
for dbid, atc in chembl_atc_results.items():
    if atc:
        per_drug_df.at[dbid, "atc_codes_list"] = atc

# Backfill targets (append to other_ids so uniprot list stays clean unless it's clearly uniprot-formatted)
for dbid, tgts in chembl_target_results.items():
    if tgts:
        existing = per_drug_df.at[dbid, "target_other_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_other_ids"] = sorted(set(existing) | set(tgts))

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags after backfill
per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x)) |
    per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = (
    per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]
)

print("After external backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

After external backfill:
has_smiles          0.972795
has_atc             0.678326
has_targets         0.776084
has_all_features    0.531241
dtype: float64


In [ ]:
# ...existing code...
# Cell 9b: Second-pass target backfill via UniProt REST API (for drugs still missing targets)

UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb/search"
UNIPROT_MAX_WORKERS = 8

def uniprot_search_by_name(query, timeout=10):
    """
    Search UniProtKB (reviewed/Swiss-Prot only) for entries matching a free-text query
    (e.g. drug name / gene name / target name). Returns list of accessions.
    """
    if not query:
        return []
    params = {
        "query": f'"{query}" AND reviewed:true',
        "fields": "accession",
        "format": "json",
        "size": 5,
    }
    try:
        r = requests.get(UNIPROT_BASE, params=params, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            results = data.get("results", [])
            return [res["primaryAccession"] for res in results if "primaryAccession" in res]
    except Exception:
        pass
    return []

def _resolve_uniprot_targets(dbid, name, other_ids):
    """
    Try resolving UniProt accessions using:
      1. drug name itself (some targets are named after drug's mechanism)
      2. any existing 'other_ids' (gene names / target names) already scraped from DrugBank/ChEMBL
    """
    found = set()

    candidates = [name] if name else []
    if isinstance(other_ids, list):
        candidates.extend(other_ids[:5])  # limit to avoid excessive querying per drug

    for cand in candidates:
        accs = uniprot_search_by_name(cand)
        found.update(accs)
        if found:
            break  # stop once we find something

    return dbid, sorted(found)

missing_targets_rows = per_drug_df.loc[
    ~per_drug_df["has_targets"],
    ["drugbank_id", "drug_name", "target_other_ids"]
]

uniprot_target_results = {}

print(f"Resolving targets via UniProt for {len(missing_targets_rows)} drugs using {UNIPROT_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=UNIPROT_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _resolve_uniprot_targets,
            row.drugbank_id,
            row.drug_name,
            row.target_other_ids
        ): row.drugbank_id
        for row in missing_targets_rows.itertuples()
    }

    completed = 0
    for future in as_completed(futures):
        dbid, accs = future.result()
        uniprot_target_results[dbid] = accs
        completed += 1

        if accs:
            print(f"[UniProt] {dbid} -> targets found: {accs}")
        if completed % 100 == 0:
            print(f"  ... {completed}/{len(missing_targets_rows)} processed")

resolved_targets_count = sum(1 for v in uniprot_target_results.values() if v)
print(f"\nUniProt resolved targets for {resolved_targets_count}/{len(uniprot_target_results)} missing-target drugs")

Resolving targets via UniProt for 749 drugs using 8 parallel workers...
[UniProt] DB00856 -> targets found: ['P06276', 'P08684', 'P11511', 'P35354', 'P47989']
[UniProt] DB00742 -> targets found: ['O93868', 'P00550', 'P0C0Y5', 'P77247', 'P77625']
[UniProt] DB01249 -> targets found: ['Q62931']
[UniProt] DB01482 -> targets found: ['P05177']
[UniProt] DB01456 -> targets found: ['P08684']
[UniProt] DB01562 -> targets found: ['P10635']
[UniProt] DB01525 -> targets found: ['A0A059Q4T4', 'E7C196', 'O00748', 'P14943', 'Q9L9D7']
[UniProt] DB01638 -> targets found: ['P0DMQ6', 'P27867', 'P56580', 'Q58D31', 'Q64442']
[UniProt] DB02224 -> targets found: ['A0A4D6Q414', 'P16559', 'Q6VMV8', 'Q6VMV9', 'Q94C57']
[UniProt] DB01563 -> targets found: ['P51647']
[UniProt] DB02245 -> targets found: ['A0A0H3KB22', 'O31675', 'O31677', 'O31678', 'O58843']
[UniProt] DB02423 -> targets found: ['A9JQL9', 'O95749', 'P60472', 'Q03426']
[UniProt] DB02520 -> targets found: ['P05181', 'P08684', 'P20815']
[UniProt] DB025

In [ ]:
# ...existing code...
# Cell 9c: Second-pass ATC backfill via WHO ATC/DDD Index (best-effort HTML scrape)
# NOTE: WHO ATC index has no official public API. This performs a lightweight scrape
# of the search results page and extracts ATC codes via regex. It is best-effort only —
# some drugs (esp. combination products, biologics-adjacent names) may not resolve.

import re

WHO_ATC_SEARCH_URL = "https://www.whocc.no/atc_ddd_index/"
WHO_MAX_WORKERS = 4  # be conservative — WHO site has no rate-limit docs, keep low
ATC_CODE_PATTERN = re.compile(r"\b([A-Z]\d{2}[A-Z]{2}\d{2})\b")

def who_atc_lookup_by_name(drug_name, timeout=10):
    """
    Query WHO ATC/DDD index search and scrape ATC codes from the results HTML.
    Returns a sorted list of unique ATC codes found (may be empty).
    """
    if not drug_name:
        return []
    params = {"name": drug_name, "showdescription": "no"}
    try:
        r = requests.get(WHO_ATC_SEARCH_URL, params=params, timeout=timeout)
        if r.status_code == 200:
            codes = set(ATC_CODE_PATTERN.findall(r.text))
            return sorted(codes)
    except Exception:
        pass
    return []

def _resolve_who_atc(dbid, name):
    codes = who_atc_lookup_by_name(name)
    return dbid, codes

missing_atc_rows = per_drug_df.loc[
    ~per_drug_df["has_atc"],
    ["drugbank_id", "drug_name"]
]

who_atc_results = {}

print(f"Resolving ATC codes via WHO ATC/DDD Index for {len(missing_atc_rows)} drugs using {WHO_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=WHO_MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_who_atc, row.drugbank_id, row.drug_name): row.drugbank_id
        for row in missing_atc_rows.itertuples()
    }

    completed = 0
    for future in as_completed(futures):
        dbid, codes = future.result()
        who_atc_results[dbid] = codes
        completed += 1

        if codes:
            print(f"[WHO ATC] {dbid} -> ATC found: {codes}")
        if completed % 50 == 0:
            print(f"  ... {completed}/{len(missing_atc_rows)} processed")

resolved_who_atc_count = sum(1 for v in who_atc_results.values() if v)
print(f"\nWHO ATC resolved codes for {resolved_who_atc_count}/{len(who_atc_results)} missing-ATC drugs")

Resolving ATC codes via WHO ATC/DDD Index for 1076 drugs using 4 parallel workers...
[WHO ATC] DB00122 -> ATC found: ['C10AB11', 'N02BA03', 'N07AX02', 'R03DA02', 'R03DB02', 'V03AB29']
[WHO ATC] DB00556 -> ATC found: ['V08DA01', 'V08DA04']
  ... 50/1076 processed
[WHO ATC] DB01369 -> ATC found: ['J01FG02']
  ... 100/1076 processed
  ... 150/1076 processed
  ... 200/1076 processed
  ... 250/1076 processed
  ... 300/1076 processed
  ... 350/1076 processed
  ... 400/1076 processed
[WHO ATC] DB08167 -> ATC found: ['V03AB17', 'V04CG05']
  ... 450/1076 processed
[WHO ATC] DB09050 -> ATC found: ['J01DI54']
[WHO ATC] DB09130 -> ATC found: ['G01AX15', 'P03AX02', 'V03AB20', 'V09IX15']
  ... 500/1076 processed
[WHO ATC] DB09401 -> ATC found: ['C01DA08', 'C01DA14', 'C01DA58', 'C05AE02']
[WHO ATC] DB11135 -> ATC found: ['D01AE13', 'D11AC03', 'V09DX01', 'V09XX03']
[WHO ATC] DB11136 -> ATC found: ['V09CX04', 'V09GX03']
  ... 550/1076 processed
  ... 600/1076 processed
  ... 650/1076 processed
[WHO ATC

In [ ]:
# ...existing code...
# Cell 9d: Merge second-pass backfill results (UniProt targets + WHO ATC) into per_drug_df

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

# Backfill targets from UniProt (add to uniprot_ids since these ARE genuine UniProt accessions)
for dbid, accs in uniprot_target_results.items():
    if accs:
        existing = per_drug_df.at[dbid, "target_uniprot_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_uniprot_ids"] = sorted(set(existing) | set(accs))

# Backfill ATC codes from WHO index
for dbid, codes in who_atc_results.items():
    if codes:
        existing = per_drug_df.at[dbid, "atc_codes_list"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "atc_codes_list"] = sorted(set(existing) | set(codes))

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags after second-pass backfill
per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x)) |
    per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = (
    per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]
)

print("After UniProt + WHO ATC second-pass backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

still_missing_smiles = per_drug_df.loc[~per_drug_df["has_smiles"], "drugbank_id"].tolist()
still_missing_atc = per_drug_df.loc[~per_drug_df["has_atc"], "drugbank_id"].tolist()
still_missing_targets = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()

print(f"\nStill missing SMILES : {len(still_missing_smiles)}")
print(f"Still missing ATC    : {len(still_missing_atc)}")
print(f"Still missing targets: {len(still_missing_targets)}")

After UniProt + WHO ATC second-pass backfill:
has_smiles          0.972795
has_atc             0.686697
has_targets         0.843647
has_all_features    0.570105
dtype: float64

Still missing SMILES : 91
Still missing ATC    : 1048
Still missing targets: 523


In [ ]:
# ...existing code...
# Cell 9e: Backfill missing FASTA sequences for target UniProt IDs — PARALLELIZED

UNIPROT_FASTA_BASE = "https://rest.uniprot.org/uniprotkb"
FASTA_MAX_WORKERS = 8

def uniprot_fetch_sequence(accession, timeout=10):
    """
    Fetch the amino-acid sequence for a single UniProt accession using the
    REST API's FASTA endpoint. Returns just the sequence body (no header).
    """
    if not accession:
        return None
    url = f"{UNIPROT_FASTA_BASE}/{accession}.fasta"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and r.text:
            lines = r.text.strip().splitlines()
            if lines and lines[0].startswith(">"):
                return "".join(lines[1:]).strip()
            return "".join(lines).strip()
    except Exception:
        pass
    return None

def _resolve_fasta_for_drug(dbid, uniprot_ids, existing_fasta):
    """
    For a single drug, fetch sequences for every UniProt accession that doesn't
    already have a matching sequence. Returns (dbid, updated_fasta_list).
    """
    existing_set = set(existing_fasta) if isinstance(existing_fasta, list) else set()

    if not isinstance(uniprot_ids, list) or not uniprot_ids:
        return dbid, sorted(existing_set)

    new_sequences = set(existing_set)
    for acc in uniprot_ids:
        seq = uniprot_fetch_sequence(acc)
        if seq:
            new_sequences.add(seq)

    return dbid, sorted(new_sequences)

# Build the unique set of UniProt accessions we need sequences for,
# by looking at drugs whose fasta list doesn't yet cover all their uniprot ids.
def _needs_fasta_backfill(row):
    uids = row["target_uniprot_ids"] if isinstance(row["target_uniprot_ids"], list) else []
    fasta = row["target_fasta_sequences"] if isinstance(row["target_fasta_sequences"], list) else []
    # crude heuristic: if we have uniprot ids but fewer (or zero) sequences, backfill
    return len(uids) > 0 and len(fasta) < len(uids)

fasta_backfill_rows = per_drug_df.loc[
    per_drug_df.apply(_needs_fasta_backfill, axis=1),
    ["drugbank_id", "target_uniprot_ids", "target_fasta_sequences"]
]

fasta_backfill_results = {}

print(f"Resolving FASTA sequences for {len(fasta_backfill_rows)} drugs using {FASTA_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=FASTA_MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _resolve_fasta_for_drug,
            row.drugbank_id,
            row.target_uniprot_ids,
            row.target_fasta_sequences
        ): row.drugbank_id
        for row in fasta_backfill_rows.itertuples()
    }

    completed = 0
    for future in as_completed(futures):
        dbid, seqs = future.result()
        fasta_backfill_results[dbid] = seqs
        completed += 1

        if completed % 100 == 0:
            print(f"  ... {completed}/{len(fasta_backfill_rows)} processed")

resolved_fasta_count = sum(
    1 for dbid, seqs in fasta_backfill_results.items()
    if len(seqs) > 0
)
print(f"\nFASTA backfill resolved sequences for {resolved_fasta_count}/{len(fasta_backfill_results)} drugs")

# Merge results back into per_drug_df
per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

for dbid, seqs in fasta_backfill_results.items():
    if seqs:
        per_drug_df.at[dbid, "target_fasta_sequences"] = seqs

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags — note: has_all_features doesn't currently
# require FASTA, but we track has_fasta separately in case you want to
# include it in the final filter later.
per_drug_df["has_fasta"] = per_drug_df["target_fasta_sequences"].apply(lambda x: not list_is_empty(x))

print("\nAfter FASTA sequence backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_fasta", "has_all_features"]].mean())

still_missing_fasta = per_drug_df.loc[~per_drug_df["has_fasta"], "drugbank_id"].tolist()
print(f"\nStill missing FASTA: {len(still_missing_fasta)}")

Resolving FASTA sequences for 236 drugs using 8 parallel workers...
  ... 100/236 processed
  ... 200/236 processed

FASTA backfill resolved sequences for 233/236 drugs

After FASTA sequence backfill:
has_smiles          0.972795
has_atc             0.686697
has_targets         0.843647
has_fasta           0.793124
has_all_features    0.570105
dtype: float64

Still missing FASTA: 692


In [ ]:
# ...existing code...
# Cell 9f: Use smpdb_protein_pathway module to backfill missing targets via
# pathway-based inference (ChEMBL -> targets -> pathways -> co-pathway proteins),
# and backfill sequences for any newly discovered targets.

# ...existing code...
import sys
sys.path.append(r"C:\Users\ashto\ddi-prediction\src_test")
import os

folder = r"C:\Users\ashto\ddi-prediction\src_test"
print("Folder exists:", os.path.exists(folder))
print("Contents:", os.listdir(folder) if os.path.exists(folder) else "N/A")

target_file = os.path.join(folder, "smpdb_protein_pathway.py")
print("Target file exists:", os.path.exists(target_file))

from smpdb_protein_pathway import (
    PathwayMapper,
    get_targets_from_chembl_batch,
    UniprotConverter,
    phi_infer_chembl_batch,
)


# Local data sources — update paths if needed
SMPDB_ZIP = r"C:\Users\ashto\ddi-prediction\data\smpdb_pathways_data_csv\smpdb_proteins.csv.zip"
DRUGBANK_XML_FOR_PATHWAYS = DRUGBANK_XML_PATH  # reuse path already defined in Cell 3

# Build a single shared PathwayMapper instance (loads SMPDB + DrugBank pathway maps once)
print("Building PathwayMapper (SMPDB + DrugBank pathway data)...")
pathway_mapper = PathwayMapper(
    xml_path=DRUGBANK_XML_FOR_PATHWAYS,
    smpdb_protein_zip=SMPDB_ZIP
)

# ── Step 1: Resolve ChEMBL molecule IDs for drugs still missing targets ──
still_missing_targets_rows = per_drug_df.loc[
    ~per_drug_df["has_targets"],
    ["drugbank_id", "drug_name"]
]

print(f"Resolving ChEMBL molecule IDs for {len(still_missing_targets_rows)} drugs still missing targets...")

def _resolve_chembl_id_only(name):
    return chembl_molecule_id_by_name(name)  # reuse function defined in Cell 8

CHEMBL_ID_MAX_WORKERS = 8
drug_to_chembl_id = {}

with ThreadPoolExecutor(max_workers=CHEMBL_ID_MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_chembl_id_only, row.drug_name): row.drugbank_id
        for row in still_missing_targets_rows.itertuples()
    }
    for future in as_completed(futures):
        dbid = futures[future]
        chembl_id = future.result()
        if chembl_id:
            drug_to_chembl_id[dbid] = chembl_id

print(f"Resolved ChEMBL IDs for {len(drug_to_chembl_id)}/{len(still_missing_targets_rows)} drugs")

# ── Step 2: Batch-resolve ChEMBL targets → UniProt accessions for all resolved IDs ──
all_chembl_ids = list(drug_to_chembl_id.values())

if all_chembl_ids:
    print(f"Querying ChEMBL mechanism/activity data for {len(all_chembl_ids)} molecules...")
    chembl_id_to_targets = get_targets_from_chembl_batch(all_chembl_ids, min_pchembl=6.0)
else:
    chembl_id_to_targets = {}

# Map back from drugbank_id -> list of UniProt accessions
pathway_backfill_targets = {}
for dbid, chembl_id in drug_to_chembl_id.items():
    targets = chembl_id_to_targets.get(chembl_id, [])
    if targets:
        pathway_backfill_targets[dbid] = targets

resolved_via_pathway_count = len(pathway_backfill_targets)
print(f"\nResolved targets via ChEMBL pathway route for {resolved_via_pathway_count} drugs")

# ── Step 3: Two-hop pathway expansion for drugs where we found direct targets ──
# This adds "neighbor" proteins (co-pathway members) as supplemental target info,
# useful for drugs with sparse direct target annotation.
all_direct_targets = set()
for targets in pathway_backfill_targets.values():
    all_direct_targets.update(targets)

print(f"\nExpanding {len(all_direct_targets)} unique direct targets to two-hop pathway neighbors...")

if all_direct_targets:
    protein_to_pathways = pathway_mapper.get_pathways_by_protein_batch(list(all_direct_targets))

    all_discovered_pathways = set()
    for pws in protein_to_pathways.values():
        all_discovered_pathways.update(pws)

    pathway_to_proteins = pathway_mapper.get_proteins_by_pathway_batch(list(all_discovered_pathways))

    all_neighbor_proteins = set()
    for prots in pathway_to_proteins.values():
        all_neighbor_proteins.update(prots)

    print(f"Discovered {len(all_discovered_pathways)} pathways, {len(all_neighbor_proteins)} neighbor proteins")
else:
    protein_to_pathways = {}
    pathway_to_proteins = {}
    all_neighbor_proteins = set()

# Build per-drug neighbor sets (drugs get neighbors from pathways their direct targets belong to)
pathway_backfill_neighbors = {}
for dbid, targets in pathway_backfill_targets.items():
    drug_pathways = set()
    for t in targets:
        drug_pathways.update(protein_to_pathways.get(t, []))

    drug_neighbors = set()
    for p in drug_pathways:
        drug_neighbors.update(pathway_to_proteins.get(p, []))

    if drug_neighbors:
        pathway_backfill_neighbors[dbid] = sorted(drug_neighbors)

print(f"\nBuilt neighbor-protein sets for {len(pathway_backfill_neighbors)} drugs")

Folder exists: True
Contents: ['atc_filter.py', 'drugbank_parse.py', 'drugbank_parse_fast.py', 'scrape_atc.py', 'smpdb_protein_pathway.py', '__init__.py', '__pycache__']
Target file exists: True
Building PathwayMapper (SMPDB + DrugBank pathway data)...
[SMPDB] Found 48687 pathway CSV files
[SMPDB] Loaded 48642 pathways
[SMPDB] Loaded 1489 proteins
[SMPDB] Loaded 303912 pathway-protein edges
[DrugBank] ZIP detected → loading: drugbank_full_database_V5.1.14.xml
Resolving ChEMBL molecule IDs for 523 drugs still missing targets...
Resolved ChEMBL IDs for 514/523 drugs
Querying ChEMBL mechanism/activity data for 514 molecules...

Resolved targets via ChEMBL pathway route for 0 drugs

Expanding 0 unique direct targets to two-hop pathway neighbors...

Built neighbor-protein sets for 0 drugs


In [ ]:
# ...existing code...
# Cell 9g: Merge pathway-based target backfill (+ neighbors) into per_drug_df

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

# Add new column for pathway-inferred neighbor proteins (supplemental feature,
# not required for has_targets — kept separate since neighbors are looser evidence
# than direct target annotations).
if "target_pathway_neighbors" not in per_drug_df.columns:
    per_drug_df["target_pathway_neighbors"] = [[] for _ in range(len(per_drug_df))]

# Backfill direct targets discovered via ChEMBL pathway route
for dbid, targets in pathway_backfill_targets.items():
    if targets:
        existing = per_drug_df.at[dbid, "target_uniprot_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_uniprot_ids"] = sorted(set(existing) | set(targets))

# Attach supplemental neighbor proteins (pathway co-membership, not direct targets)
for dbid, neighbors in pathway_backfill_neighbors.items():
    if neighbors:
        existing = per_drug_df.at[dbid, "target_pathway_neighbors"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_pathway_neighbors"] = sorted(set(existing) | set(neighbors))

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags
per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x)) |
    per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = (
    per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]
)

print("After pathway-based target backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

still_missing_targets_after_pathway = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()
print(f"\nStill missing targets: {len(still_missing_targets_after_pathway)}")

After pathway-based target backfill:
has_smiles          0.972795
has_atc             0.686697
has_targets         0.843647
has_all_features    0.570105
dtype: float64

Still missing targets: 523


In [ ]:
# ...existing code...
# Cell 9g: Merge pathway-based target backfill (+ neighbors) into per_drug_df

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

# Add new column for pathway-inferred neighbor proteins (supplemental feature,
# not required for has_targets — kept separate since neighbors are looser evidence
# than direct target annotations).
if "target_pathway_neighbors" not in per_drug_df.columns:
    per_drug_df["target_pathway_neighbors"] = [[] for _ in range(len(per_drug_df))]

# Backfill direct targets discovered via ChEMBL pathway route
for dbid, targets in pathway_backfill_targets.items():
    if targets:
        existing = per_drug_df.at[dbid, "target_uniprot_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_uniprot_ids"] = sorted(set(existing) | set(targets))

# Attach supplemental neighbor proteins (pathway co-membership, not direct targets)
for dbid, neighbors in pathway_backfill_neighbors.items():
    if neighbors:
        existing = per_drug_df.at[dbid, "target_pathway_neighbors"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_pathway_neighbors"] = sorted(set(existing) | set(neighbors))

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags
per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x)) |
    per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = (
    per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]
)

print("After pathway-based target backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

still_missing_targets_after_pathway = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()
print(f"\nStill missing targets: {len(still_missing_targets_after_pathway)}")

After pathway-based target backfill:
has_smiles          0.972795
has_atc             0.686697
has_targets         0.843647
has_all_features    0.570105
dtype: float64

Still missing targets: 523


In [ ]:
# ...existing code...
# Cell 9h: Batch-fetch FASTA sequences (+ GO terms, Pfam domains) for ANY newly
# discovered UniProt targets — using the UniprotConverter class for efficient
# batched retrieval (chunks of 100 IDs per call, single cached fetch per instance).

# Collect every UniProt accession across the WHOLE per_drug_df dataset that
# still lacks a matching FASTA sequence (covers both original DrugBank targets
# and the newly pathway-backfilled ones from Cell 9g).

def _get_missing_fasta_accessions(row):
    uids = row.target_uniprot_ids if isinstance(row.target_uniprot_ids, list) else []
    fasta = row.target_fasta_sequences if isinstance(row.target_fasta_sequences, list) else []
    if not uids:
        return []
    # crude heuristic: if fewer sequences than accessions, assume some are missing
    if len(fasta) >= len(uids):
        return []
    return uids

all_missing_fasta_accessions = set()
for row in per_drug_df.itertuples():
    accs = _get_missing_fasta_accessions(row)
    all_missing_fasta_accessions.update(accs)

print(f"Batch-fetching sequences/GO/Pfam for {len(all_missing_fasta_accessions)} unique UniProt accessions...")

if all_missing_fasta_accessions:
    batch_converter = UniprotConverter(list(all_missing_fasta_accessions))

    accession_to_sequence = batch_converter.uniprot_to_sequence()
    accession_to_go = batch_converter.uniprot_to_GO_terms()
    accession_to_pfam = batch_converter.uniprot_to_pfams()

    resolved_seq_count = sum(1 for s in accession_to_sequence.values() if s)
    print(f"Resolved {resolved_seq_count}/{len(all_missing_fasta_accessions)} sequences")
else:
    accession_to_sequence = {}
    accession_to_go = {}
    accession_to_pfam = {}

# ── Merge sequences back into per_drug_df ──
per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

if "target_go_terms" not in per_drug_df.columns:
    per_drug_df["target_go_terms"] = [[] for _ in range(len(per_drug_df))]
if "target_pfam_domains" not in per_drug_df.columns:
    per_drug_df["target_pfam_domains"] = [[] for _ in range(len(per_drug_df))]

for row in per_drug_df.itertuples():
    dbid = row.drugbank_id
    uids = row.target_uniprot_ids if isinstance(row.target_uniprot_ids, list) else []

    if not uids:
        continue

    # Sequences
    existing_fasta = row.target_fasta_sequences if isinstance(row.target_fasta_sequences, list) else []
    new_sequences = set(existing_fasta)
    for acc in uids:
        seq = accession_to_sequence.get(acc, "")
        if seq:
            new_sequences.add(seq)
    if new_sequences != set(existing_fasta):
        per_drug_df.at[dbid, "target_fasta_sequences"] = sorted(new_sequences)

    # GO terms
    existing_go = row.target_go_terms if isinstance(row.target_go_terms, list) else []
    new_go = set(existing_go)
    for acc in uids:
        new_go.update(accession_to_go.get(acc, []))
    if new_go != set(existing_go):
        per_drug_df.at[dbid, "target_go_terms"] = sorted(new_go)

    # Pfam domains
    existing_pfam = row.target_pfam_domains if isinstance(row.target_pfam_domains, list) else []
    new_pfam = set(existing_pfam)
    for acc in uids:
        new_pfam.update(accession_to_pfam.get(acc, []))
    if new_pfam != set(existing_pfam):
        per_drug_df.at[dbid, "target_pfam_domains"] = sorted(new_pfam)

per_drug_df = per_drug_df.reset_index(drop=True)

# Recompute completeness flags
per_drug_df["has_fasta"] = per_drug_df["target_fasta_sequences"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_go_terms"] = per_drug_df["target_go_terms"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_pfam"] = per_drug_df["target_pfam_domains"].apply(lambda x: not list_is_empty(x))

print("\nAfter batch UniProt sequence/GO/Pfam backfill:")
print(per_drug_df[[
    "has_smiles", "has_atc", "has_targets", "has_fasta",
    "has_go_terms", "has_pfam", "has_all_features"
]].mean())

Batch-fetching sequences/GO/Pfam for 368 unique UniProt accessions...
Resolved 367/368 sequences

After batch UniProt sequence/GO/Pfam backfill:
has_smiles          0.972795
has_atc             0.686697
has_targets         0.843647
has_fasta           0.794021
has_go_terms        0.137220
has_pfam            0.137220
has_all_features    0.570105
dtype: float64


In [ ]:
# ...existing code...
# Cell 9i: Two-hop pathway neighbor expansion for ALL drugs' existing targets
# (not just the pathway-backfilled ones from Cell 9f/9g). This uses the same
# guilt-by-association logic as phi_infer_batch() in smpdb_protein_pathway.py:
#   Hop 1: protein -> pathways it belongs to
#   Hop 2: those pathways -> all co-member proteins (neighbors)
# Reuses the shared `pathway_mapper` instance built in Cell 9f (no need to
# reload SMPDB/DrugBank data again).

# Collect the full universe of UniProt accessions currently present across
# every drug's target_uniprot_ids (original + all backfilled sources).
all_current_targets = set()
for row in per_drug_df.itertuples():
    uids = row.target_uniprot_ids if isinstance(row.target_uniprot_ids, list) else []
    all_current_targets.update(uids)

print(f"Expanding {len(all_current_targets)} unique target proteins (full dataset) to two-hop pathway neighbors...")

if all_current_targets:
    # Hop 1: protein -> pathways
    full_protein_to_pathways = pathway_mapper.get_pathways_by_protein_batch(list(all_current_targets))

    all_discovered_pathways_full = set()
    for pws in full_protein_to_pathways.values():
        all_discovered_pathways_full.update(pws)

    # Hop 2: pathways -> co-member proteins
    full_pathway_to_proteins = pathway_mapper.get_proteins_by_pathway_batch(list(all_discovered_pathways_full))

    all_neighbor_proteins_full = set()
    for prots in full_pathway_to_proteins.values():
        all_neighbor_proteins_full.update(prots)

    print(f"Discovered {len(all_discovered_pathways_full)} pathways, {len(all_neighbor_proteins_full)} neighbor proteins (dataset-wide)")
else:
    full_protein_to_pathways = {}
    full_pathway_to_proteins = {}
    all_neighbor_proteins_full = set()

# Build per-drug neighbor sets using each drug's OWN existing targets
full_pathway_neighbors_by_drug = {}
for row in per_drug_df.itertuples():
    dbid = row.drugbank_id
    uids = row.target_uniprot_ids if isinstance(row.target_uniprot_ids, list) else []
    if not uids:
        continue

    drug_pathways = set()
    for t in uids:
        drug_pathways.update(full_protein_to_pathways.get(t, []))

    drug_neighbors = set()
    for p in drug_pathways:
        drug_neighbors.update(full_pathway_to_proteins.get(p, []))

    if drug_neighbors:
        full_pathway_neighbors_by_drug[dbid] = sorted(drug_neighbors)

print(f"\nBuilt dataset-wide neighbor-protein sets for {len(full_pathway_neighbors_by_drug)} drugs")

# ── Merge into per_drug_df ──
per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

if "target_pathway_neighbors" not in per_drug_df.columns:
    per_drug_df["target_pathway_neighbors"] = [[] for _ in range(len(per_drug_df))]

for dbid, neighbors in full_pathway_neighbors_by_drug.items():
    existing = per_drug_df.at[dbid, "target_pathway_neighbors"]
    existing = existing if isinstance(existing, list) else []
    per_drug_df.at[dbid, "target_pathway_neighbors"] = sorted(set(existing) | set(neighbors))

per_drug_df = per_drug_df.reset_index(drop=True)

per_drug_df["has_pathway_neighbors"] = per_drug_df["target_pathway_neighbors"].apply(lambda x: not list_is_empty(x))

print("\nAfter dataset-wide pathway neighbor expansion:")
print(per_drug_df[[
    "has_smiles", "has_atc", "has_targets", "has_fasta",
    "has_go_terms", "has_pfam", "has_pathway_neighbors", "has_all_features"
]].mean())
# ...existing code...

Expanding 2893 unique target proteins (full dataset) to two-hop pathway neighbors...
Discovered 49832 pathways, 9584 neighbor proteins (dataset-wide)

Built dataset-wide neighbor-protein sets for 2425 drugs

After dataset-wide pathway neighbor expansion:
has_smiles               0.972795
has_atc                  0.686697
has_targets              0.843647
has_fasta                0.794021
has_go_terms             0.137220
has_pfam                 0.137220
has_pathway_neighbors    0.724963
has_all_features         0.570105
dtype: float64


In [ ]:
######################################################################

In [ ]:
# ...existing code...
# Cell 10: NOW discard drugs missing any feature — from BOTH datasets
# Keep track of approved vs unapproved counts before AND after this final cut

# Extend feature_cols to include the enriched pathway/GO/Pfam columns so they
# get attached to the final pair-level datasets too.
final_feature_cols = feature_cols + [
    "target_go_terms",
    "target_pfam_domains",
    "target_pathway_neighbors",
]

complete_drug_ids = set(per_drug_df.loc[per_drug_df["has_all_features"], "drugbank_id"])
approved_lookup = per_drug_df.set_index("drugbank_id")["is_approved"].to_dict()

print(f"Drugs with all features complete: {len(complete_drug_ids)} / {len(per_drug_df)}")

# Breakdown of the complete-feature pool by approval status
complete_approved = sum(1 for d in complete_drug_ids if approved_lookup.get(d, False))
complete_unapproved = len(complete_drug_ids) - complete_approved
print(f"  -> Approved   : {complete_approved}")
print(f"  -> Unapproved : {complete_unapproved}")

negative_final_df = filter_pairs_to_ids(negative_df, neg_d1, neg_d2, complete_drug_ids)
adverse_final_df  = filter_pairs_to_ids(adverse_df,  adv_d1, adv_d2, complete_drug_ids)

print(f"\nNegative: {len(negative_df)} -> {len(negative_final_df)}")
print(f"Adverse : {len(adverse_df)} -> {len(adverse_final_df)}")

# Re-attach final features cleanly, including the pathway/GO/Pfam enrichment columns
negative_final_df = attach_features(
    negative_final_df, neg_d1, neg_d2,
    per_drug_df.set_index("drugbank_id"), final_feature_cols
)
adverse_final_df = attach_features(
    adverse_final_df, adv_d1, adv_d2,
    per_drug_df.set_index("drugbank_id"), final_feature_cols
)

def unique_drug_ids(df, d1_col, d2_col):
    return set(
        pd.concat([df[d1_col], df[d2_col]]).astype(str).str.strip().str.upper().unique()
    )

neg_final_ids = unique_drug_ids(negative_final_df, neg_d1, neg_d2)
adv_final_ids = unique_drug_ids(adverse_final_df, adv_d1, adv_d2)
union_final_ids = neg_final_ids | adv_final_ids

def approved_unapproved_split(ids, lookup):
    approved = sum(1 for d in ids if lookup.get(d, False))
    unapproved = len(ids) - approved
    return approved, unapproved

neg_app, neg_unapp = approved_unapproved_split(neg_final_ids, approved_lookup)
adv_app, adv_unapp = approved_unapproved_split(adv_final_ids, approved_lookup)
union_app, union_unapp = approved_unapproved_split(union_final_ids, approved_lookup)

print(f"\nUnique drugs in final NEGATIVE dataset : {len(neg_final_ids)}  (Approved: {neg_app}, Unapproved: {neg_unapp})")
print(f"Unique drugs in final ADVERSE  dataset : {len(adv_final_ids)}  (Approved: {adv_app}, Unapproved: {adv_unapp})")
print(f"Unique drugs across BOTH (union)       : {len(union_final_ids)}  (Approved: {union_app}, Unapproved: {union_unapp})")

# Quick sanity check: confirm the new enrichment columns made it into the final pair datasets
enrichment_check_cols = [c for c in negative_final_df.columns if c.startswith("target_pathway_neighbors")]
print(f"\nEnrichment columns present in negative_final_df: {enrichment_check_cols}")

negative_final_df.head()
# ...existing code...

Drugs with all features complete: 1907 / 3345
  -> Approved   : 1579
  -> Unapproved : 328

Negative: 569934 -> 162893
Adverse : 830623 -> 478324

Unique drugs in final NEGATIVE dataset : 1902  (Approved: 1578, Unapproved: 324)
Unique drugs in final ADVERSE  dataset : 1900  (Approved: 1573, Unapproved: 327)
Unique drugs across BOTH (union)       : 1907  (Approved: 1579, Unapproved: 328)

Enrichment columns present in negative_final_df: ['target_pathway_neighbors_1', 'target_pathway_neighbors_2']


,drug1_id,drug1_name,drug2_id,drug2_name,description,smiles_1,smiles_2,atc_codes_list_1,atc_codes_list_2,target_uniprot_ids_1,...,target_fasta_sequences_1,target_fasta_sequences_2,target_other_ids_1,target_other_ids_2,target_go_terms_1,target_go_terms_2,target_pfam_domains_1,target_pfam_domains_2,target_pathway_neighbors_1,target_pathway_neighbors_2
0,DB06608,Tafenoquine,DB08899,Enzalutamide,No documented interaction,COC1=CC(C)=C2C(OC3=CC=CC(=C3)C(F)(F)F)=C(OC)C=...,CNC(=O)C1=C(F)C=C(C=C1)N1C(=S)N(C(=O)C1(C)C)C1...,[P01BA07],[L02BB04],"[C6KT68, O15244, P10635, Q86VL8, Q96FL8]",...,[MDSLQDTVALDHGGCCPALSRLVPRGFGTEMWTLFALSGPLFLFQ...,[MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAA...,[],"[178628, 628, ANDR_HUMAN, AR, Androgen recepto...","[GO:0005275, GO:0005277, GO:0005326, GO:000588...",[],"[PF00083, PF01554]",[],"[A4Z6T9, A8E194, A9YTQ3, B7ZKV8, O00180, O0030...","[A6NCW0, A6NCW7, A8MUK1, B7ZKV8, C9J2P7, C9JJH..."
16,DB01132,Pioglitazone,DB16019,Gallium Ga-68 gozetotide,No documented interaction,CCC1=CN=C(CCOC2=CC=C(CC3SC(=O)NC3=O)C=C2)C=C1,[68Ga+3].OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O...,"[A10BD05, A10BD06, A10BD09, A10BD12, A10BG03]",[V09IX14],"[P27338, P37231]",...,[MGETLGDSPIDPESDSFTDTLSANISQEMTMVDTEMPFWPTNFGI...,[MWNLLHETDSAVATARRPRWLCAGALVLAGGFFLLGFLFGWFIKS...,"[1711117, 2490, 398415, 595, AOFB_HUMAN, Amine...","[1606, 190664, FOLH1, FOLH1_HUMAN, Glutamate c...",[],[],[],[],"[A0JLT2, A9YTQ3, B7ZKV8, D6RB81, O00141, O0018...","[A0A5P8YJZ1, C7C422, H0YLL5, O00222, O00341, O..."
20,DB00442,Entecavir,DB00652,Pentazocine,No documented interaction,NC1=NC(=O)C2=C(N1)N(C=N2)[C@H]1C[C@H](O)[C@@H]...,CC1C2CC3=C(C=C(O)C=C3)C1(C)CCN2CC=C(C)C,[J05AF10],"[N02AD01, N02AD51]",[Q3MS49],...,[MGVGLSPFLLSQFTSAICSVVRRAFPHCLAFSYMDDVVLGAKTV],[MDSPIQIFRGEPGPTCAPSACLPPNSSAWFPGWAEPDSNGSAGSE...,"[Q3MS49, Q3MS49_HBV, Reverse transcriptase, rt]","[1783387, 2552, 318, 319, 452073, 532060, HGNC...",[],[],[],[],[],"[B7ZKV8, O00206, O00230, O00254, O00421, O0055..."
23,DB08933,Luliconazole,DB13559,Rimiterol,No documented interaction,ClC1=CC(Cl)=C(C=C1)[C@@H]1CS\C(S1)=C(\C#N)N1C=...,O[C@@H]([C@@H]1CCCCN1)C1=CC(O)=C(O)C=C1,[D01AC18],[R03AC05],[P10613],...,[MAIVETVIDGINYFLSLSVTQQISILLGVPFVYNLVWQYLYSLRK...,[MAPWPHENSSLAPWPDLPTLAPNTANTSGLPGVPWEAALAGALLA...,"[578119, CP51_CANAL, ERG11, Lanosterol 14-alph...","[178896, 30, ADRB3, ADRB3_HUMAN, Beta-3 adrene...",[],[],[],[],"[A0A5P8YJZ1, C7C422, H0YLL5, O00222, O00341, O...","[A8MTJ3, B7ZKV8, O00155, O00180, O00305, O0033..."
24,DB01576,Dextroamphetamine,DB11796,Fostemsavir,No documented interaction,C[C@H](N)CC1=CC=CC=C1,COC1=CN=C(N2C=NC(C)=N2)C2=C1C(=CN2COP(O)(O)=O)...,[N06BA02],[J05AX29],"[P23975, P35348, P35368, Q01959, Q05940, Q96RJ0]",...,[MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIP...,[MRVKGIKKNYQHLWRWGGMMLLGILMICSATDKLWVTVYYGVPVW...,"[1012, 14600074, 189258, 22, 23, 292335, 364, ...","[ENV_HV1BN, Envelope glycoprotein gp160, M2109...","[GO:0001994, GO:0004937, GO:0005634, GO:000573...",[],[PF00001],[],"[A1IGU5, A5YM69, A8MTJ3, A8MVX0, B7ZKV8, O0015...",[]


In [ ]:
adverse_final_df.head()

,drug1_id,drug1_name,drug2_id,drug2_name,description,pair_key,smiles_1,smiles_2,atc_codes_list_1,atc_codes_list_2,...,target_fasta_sequences_1,target_fasta_sequences_2,target_other_ids_1,target_other_ids_2,target_go_terms_1,target_go_terms_2,target_pfam_domains_1,target_pfam_domains_2,target_pathway_neighbors_1,target_pathway_neighbors_2
0,DB00006,Bivalirudin,DB06605,Apixaban,Apixaban may increase the anticoagulant activi...,"('DB00006', 'DB06605')",CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,COC1=CC=C(C=C1)N1N=C(C(N)=O)C2=C1C(=O)N(CC2)C1...,[B01AE06],[B01AF02],...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MGRPLHLVLLSASLAGLLLLGESLFIRREQANNILARVTRANSFL...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[182841, 2359, Coagulation factor X, F10, FA10...",[],[],[],[],"[O00206, O00230, O00254, O00421, O00468, O0057...","[O14733, O43561, O75469, O95750, P00451, P0048..."
1,DB00006,Bivalirudin,DB06695,Dabigatran etexilate,Dabigatran etexilate may increase the anticoag...,"('DB00006', 'DB06695')",CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,CCCCCCOC(=O)\N=C(\N)C1=CC=C(NCC2=NC3=C(C=CC(=C...,[B01AE06],[B01AE07],...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[2362, 339641, F2, HGNC:3535, M17262, Prothrom...",[],[],[],[],"[O00206, O00230, O00254, O00421, O00468, O0057...","[O00206, O00230, O00254, O00421, O00468, O0057..."
2,DB00006,Bivalirudin,DB01254,Dasatinib,The risk or severity of bleeding and hemorrhag...,"('DB00006', 'DB01254')",CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,CC1=NC(NC2=NC=C(S2)C(=O)NC2=C(C)C=CC=C2Cl)=CC(...,[B01AE06],[L01EA02],...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFE...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[10635153, 11526573, 1499, 178993, 1804, 1805,...",[],"[GO:0000398, GO:0000974, GO:0001664, GO:000372...",[],[PF00012],"[O00206, O00230, O00254, O00421, O00468, O0057...","[A0A075B6P5, A0A075B6S6, A0A0A6YYK7, A0A0C4DH2..."
3,DB00006,Bivalirudin,DB01609,Deferasirox,The risk or severity of gastrointestinal bleed...,"('DB00006', 'DB01609')",CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,OC(=O)C1=CC=C(C=C1)N1N=C(N=C1C1=CC=CC=C1O)C1=C...,[B01AE06],[V03AC03],...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[],"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...",[Iron],[],[],[],[],"[O00206, O00230, O00254, O00421, O00468, O0057...",[]
4,DB00006,Bivalirudin,DB01586,Ursodeoxycholic acid,The risk or severity of bleeding and bruising ...,"('DB00006', 'DB01586')",CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,[H][C@@]1(CC[C@@]2([H])[C@]3([H])[C@@H](O)C[C@...,[B01AE06],[A05AA02],...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MDSKYQCVKLNDGHFMPVLGFGTYAPAEVPKSKALEAVKLAIEAG...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[1246749, 1546084, 531160, 603, AK1C2_HUMAN, A...",[],[],[],[],"[O00206, O00230, O00254, O00421, O00468, O0057...","[A0JLT2, A8MV81, A9YTQ3, H0YLL5, O00217, O0032..."


In [ ]:
adverse_final_df.to_parquet("C:/Users/ashto/ddi-prediction/data/sample/adverse_final_df.parquet", index=False, compression="snappy")
negative_final_df.to_parquet("C:/Users/ashto/ddi-prediction/data/sample/negative_final_df.parquet", index=False, compression="snappy")